# Software FJ – Versión integrada
Sistema Integral de Gestión de Clientes, Servicios y Reservas (UNAD – Programación 213023).

Este notebook contiene el **mismo código** de los archivos `.py` del proyecto, un módulo por celda.
Ejecuta las celdas **de arriba hacia abajo**; la última celda corre la simulación completa.
Los logs se guardan en la carpeta `logs/` (junto al notebook).

## Módulo `excepciones.py`

In [ ]:
"""
Módulo: excepciones.py
ORIGEN: Proyecto 1 (jerarquía completa de excepciones y el flag "registrada").
        Los casos de error del Proyecto 2 (servicio no disponible, cálculo
        inconsistente, datos inválidos) se cubren con estas mismas clases.

Una "excepción personalizada" es un tipo de error propio del sistema.
Así, en vez de un error genérico, el programa dice exactamente QUÉ salió mal.

Árbol de excepciones:

    SoftwareFJError                      (error base del sistema)
    ├── ClienteInvalidoError             (datos de cliente incorrectos)
    │   └── ClienteDuplicadoError        (documento ya registrado)
    ├── ServicioInvalidoError            (servicio mal creado)
    ├── ParametroInvalidoError           (un parámetro tiene un valor malo)
    │   └── ParametroFaltanteError       (falta un parámetro obligatorio)
    ├── ServicioNoDisponibleError        (el servicio está ocupado o apagado)
    ├── CalculoInconsistenteError        (el costo no tiene sentido)
    ├── ReservaError                     (la reserva no se pudo crear/cambiar)
    └── OperacionNoPermitidaError        (acción prohibida en ese estado)
"""


class SoftwareFJError(Exception):
    """Error base: todas las demás excepciones del sistema heredan de esta.
    Gracias a eso, con un solo `except SoftwareFJError` se atrapa cualquiera."""

    # Marca para no escribir el mismo error dos veces en el archivo de logs.
    # False = "todavía no se ha escrito en el log".
    registrada = False


class ClienteInvalidoError(SoftwareFJError):
    """Los datos del cliente (nombre, documento, correo, teléfono) son malos."""


class ClienteDuplicadoError(ClienteInvalidoError):
    """Se intentó registrar un cliente con un documento que ya existe."""


class ServicioInvalidoError(SoftwareFJError):
    """El servicio se creó mal (tarifa negativa, nombre vacío, etc.)."""


class ParametroInvalidoError(SoftwareFJError):
    """Un parámetro existe pero su valor no sirve (ej.: duración negativa)."""


class ParametroFaltanteError(ParametroInvalidoError):
    """No se envió un parámetro obligatorio (ej.: 'asistentes' en una sala)."""


class ServicioNoDisponibleError(SoftwareFJError):
    """El servicio no se puede reservar porque está ocupado o desactivado."""


class CalculoInconsistenteError(SoftwareFJError):
    """El cálculo del costo da algo ilógico (descuento fuera de tope, total <= 0)."""


class ReservaError(SoftwareFJError):
    """No se pudo crear, confirmar, cancelar o procesar una reserva."""


class OperacionNoPermitidaError(SoftwareFJError):
    """La acción no se puede hacer en el estado actual (ej.: procesar una
    reserva que ya está cancelada)."""

## Módulo `entidad.py`

In [ ]:
"""
Módulo: entidad.py
ORIGEN: Proyecto 1 -> IDs automáticos (CLI-001, SRV-001, RES-001) con contador
                      privado por prefijo y método abstracto mostrar_datos().
        Proyecto 2 -> método abstracto validar_integridad() y obtener_resumen().

EntidadBase es una clase ABSTRACTA: no se puede crear directamente
(EntidadBase() daría error). Solo sirve como "molde" para Cliente, Servicio
y Reserva, que heredan de ella.
"""

from abc import ABC, abstractmethod  # ABC = "Abstract Base Class" (clase abstracta)


class EntidadBase(ABC):
    """Molde común de todo lo que tiene un ID en el sistema."""

    # Cada clase hija cambia este prefijo (CLI, SRV, RES) para armar su ID.
    _PREFIJO = "ENT"

    # Diccionario PRIVADO y compartido por todas las clases: guarda cuántos IDs
    # se han entregado por prefijo. Ejemplo: {"CLI": 2, "SRV": 4}
    __contadores = {}

    def __init__(self):
        # Busca el último número usado para este prefijo (0 si es el primero) y suma 1
        numero = EntidadBase.__contadores.get(self._PREFIJO, 0) + 1
        # Guarda el nuevo número para que el siguiente objeto no lo repita
        EntidadBase.__contadores[self._PREFIJO] = numero
        # Arma el ID final con 3 dígitos: "CLI-001", "CLI-002", ...
        self.__id = f"{self._PREFIJO}-{numero:03d}"

    @property
    def id(self):
        """El ID es de SOLO LECTURA: se puede consultar, pero no cambiar."""
        return self.__id

    @abstractmethod
    def mostrar_datos(self):
        """Cada clase hija debe devolver un texto con sus datos."""

    @abstractmethod
    def validar_integridad(self):
        """Cada clase hija debe poder decir si sus datos internos son coherentes
        (devuelve True/False)."""

    def obtener_resumen(self):
        """Resumen corto de la entidad. Por defecto es lo mismo que mostrar_datos()."""
        return self.mostrar_datos()

    def __str__(self):
        # Cuando se hace print(objeto) se muestra el texto de mostrar_datos()
        return self.mostrar_datos()

## Módulo `logger_config.py`

In [ ]:
"""
Módulo: logger_config.py
ORIGEN: Proyecto 1 (clase Logger + decorador @registrar_errores + marca
        "registrada" para no duplicar errores).
        Se mantiene el archivo de log con el nombre que usan ambos
        proyectos: logs/software_fj.log

El "log" es un archivo de texto donde el programa anota todo lo que pasa
(eventos, advertencias y errores) con fecha y hora. Sirve para revisar
después qué ocurrió. Es el ÚNICO uso de archivos del sistema (no hay base de datos).
"""

import functools   # Sirve para que los decoradores no "disfracen" el nombre de la función
import logging     # Módulo de Python para escribir logs
import os          # Para manejar rutas de carpetas y archivos


# Carpeta donde está este archivo. Si el código se ejecuta dentro de un
# Notebook (Jupyter), __file__ no existe y se usa la carpeta actual.
try:
    _CARPETA_BASE = os.path.dirname(os.path.abspath(__file__))
except NameError:
    _CARPETA_BASE = os.getcwd()


class Logger:
    """Envoltura sencilla sobre el módulo logging de Python."""

    # Ruta completa del archivo de log: <carpeta del proyecto>/logs/software_fj.log
    RUTA_LOG = os.path.join(_CARPETA_BASE, "logs", "software_fj.log")

    # Aquí se guarda el logger ya creado, para crearlo UNA sola vez
    _logger = None

    @classmethod
    def configurar_logger(cls, ruta=None, nivel=logging.INFO):
        """Crea el logger la primera vez; las siguientes veces reutiliza el mismo."""
        if cls._logger is not None:      # Si ya existe, lo devuelve sin crear otro
            return cls._logger

        ruta = ruta or cls.RUTA_LOG      # Si no dan ruta, usa la de por defecto
        logger = logging.getLogger("software_fj")
        logger.setLevel(nivel)           # Nivel mínimo que se escribe (INFO en adelante)
        logger.propagate = False         # Evita que el mensaje salga repetido en consola

        # Formato de cada línea: fecha y hora | nivel | mensaje
        formato = logging.Formatter("%(asctime)s | %(levelname)-7s | %(message)s")
        try:
            os.makedirs(os.path.dirname(ruta), exist_ok=True)   # Crea la carpeta /logs si falta
            manejador = logging.FileHandler(ruta, encoding="utf-8")  # Escribe en el archivo
        except OSError:
            # Si no se puede escribir el archivo, el sistema NO se detiene:
            # usa la consola como plan B.
            manejador = logging.StreamHandler()
        manejador.setFormatter(formato)
        logger.addHandler(manejador)

        cls._logger = logger
        return logger

    @classmethod
    def registrar_evento(cls, mensaje):
        """Anota algo normal que pasó (nivel INFO)."""
        cls.configurar_logger().info(mensaje)

    @classmethod
    def registrar_advertencia(cls, mensaje):
        """Anota algo que merece atención pero no es grave (nivel WARNING)."""
        cls.configurar_logger().warning(mensaje)

    @classmethod
    def registrar_error(cls, mensaje, excepcion=None):
        """Anota un error (nivel ERROR). Si se envía la excepción, también deja
        escrita la cadena de causas (los errores encadenados con 'raise ... from')."""
        detalle = mensaje
        if excepcion is not None:
            # Agrega el tipo de error y su texto
            detalle += f" | {type(excepcion).__name__}: {excepcion}"
            # Recorre las causas: __cause__ guarda el error original de un "raise ... from"
            causa = excepcion.__cause__
            while causa is not None:
                detalle += f" <- causa: {type(causa).__name__}: {causa}"
                causa = causa.__cause__
            # Marca el error como "ya registrado" para no escribirlo dos veces
            if isinstance(excepcion, SoftwareFJError):
                excepcion.registrada = True
        cls.configurar_logger().error(detalle)


def registrar_errores(funcion):
    """DECORADOR. Se coloca encima de una función con @registrar_errores.
    Si la función lanza un error del sistema, el decorador lo anota en el log
    y luego lo vuelve a lanzar SIN cambiarlo (el programa sigue su flujo normal)."""
    @functools.wraps(funcion)            # Conserva el nombre original de la función
    def envoltura(*args, **kwargs):
        try:
            return funcion(*args, **kwargs)      # Ejecuta la función normalmente
        except SoftwareFJError as error:
            if not error.registrada:             # Solo lo anota si nadie lo anotó antes
                Logger.registrar_error(f"Error en {funcion.__qualname__}", error)
            raise                                # Relanza el mismo error intacto
    return envoltura

## Módulo `cliente.py`

In [ ]:
"""
Módulo: cliente.py
ORIGEN: Proyecto 1 -> firma del constructor (nombre, documento, correo), datos
                      privados, getters/setters y validaciones estrictas con
                      expresiones regulares; acumula TODOS los errores juntos.
        Proyecto 2 -> el campo TELÉFONO (en el Proyecto 1 no existía).
                      Aquí es opcional para no romper la firma del Proyecto 1,
                      pero si se entrega, se valida.
"""

import re  # Módulo de "expresiones regulares": sirve para revisar el formato de un texto



class Cliente(EntidadBase):
    """Cliente de Software FJ. Sus datos personales son PRIVADOS (doble guion
    bajo): desde afuera solo se leen o cambian con getters/setters, que SIEMPRE
    validan. Así nadie puede guardar un dato malo saltándose las reglas."""

    _PREFIJO = "CLI"  # Los IDs de cliente serán CLI-001, CLI-002...

    # Letras permitidas en un nombre (incluye tildes, ñ y ü)
    _LETRAS = "A-Za-zÁÉÍÓÚÜÑáéíóúüñ"
    # Nombre: solo letras, y las palabras separadas por un espacio
    _PATRON_NOMBRE = re.compile(rf"^[{_LETRAS}]+(?: [{_LETRAS}]+)*$")
    # Documento: entre 6 y 12 dígitos
    _PATRON_DOCUMENTO = re.compile(r"^\d{6,12}$")
    # Correo: formato usuario@dominio.ext
    _PATRON_CORREO = re.compile(r"^[\w.+-]+@[\w-]+(?:\.[\w-]+)*\.[A-Za-z]{2,}$")
    # Teléfono (del Proyecto 2): entre 7 y 15 dígitos
    _PATRON_TELEFONO = re.compile(r"^\d{7,15}$")

    @registrar_errores  # Si falla, el error queda escrito en el log
    def __init__(self, nombre, documento, correo, telefono=None):
        # Se validan los datos uno por uno, pero sin detenerse en el primer error:
        # se juntan todos para avisarle al usuario TODO lo que está mal de una vez.
        errores = []
        for setter, valor in ((self.set_nombre, nombre),
                              (self.set_documento, documento),
                              (self.set_correo, correo),
                              (self.set_telefono, telefono)):
            try:
                setter(valor)                       # Intenta guardar el dato (valida adentro)
            except ClienteInvalidoError as error:
                errores.append(error)               # Si falla, guarda el error en la lista

        if errores:
            # Encadenamiento: el error general "viene de" el primer error específico.
            # Por eso en el log se ve el mensaje general y también la causa original.
            raise ClienteInvalidoError(
                "No se pudo crear el cliente: " + " | ".join(str(e) for e in errores)
            ) from errores[0]

        super().__init__()  # Asigna el ID (CLI-001...) solo si el cliente es válido

    # ---------------- Validaciones ----------------
    @classmethod
    def validar_nombre(cls, valor):
        """Revisa el nombre y lo devuelve limpio (mínimo 3 letras, sin números ni símbolos)."""
        if not isinstance(valor, str):
            raise ClienteInvalidoError("El nombre debe ser texto")
        limpio = " ".join(valor.split())  # Quita espacios sobrantes ("Ana   Paz " -> "Ana Paz")
        if len(limpio) < 3 or not cls._PATRON_NOMBRE.match(limpio):
            raise ClienteInvalidoError(
                f"Nombre inválido '{valor}': mínimo 3 letras, sin números ni símbolos")
        return limpio

    @classmethod
    def validar_documento(cls, valor):
        """Revisa el documento y lo devuelve como texto (6 a 12 dígitos)."""
        # bool se descarta aparte porque en Python True/False cuentan como números
        if isinstance(valor, bool) or not isinstance(valor, (str, int)):
            raise ClienteInvalidoError("El documento debe ser texto o número")
        limpio = str(valor).strip()  # Se guarda siempre como texto
        if not cls._PATRON_DOCUMENTO.match(limpio):
            raise ClienteInvalidoError(
                f"Documento inválido '{valor}': debe tener entre 6 y 12 dígitos")
        return limpio

    @classmethod
    def validar_correo(cls, valor):
        """Revisa el correo y lo devuelve en minúsculas."""
        if not isinstance(valor, str):
            raise ClienteInvalidoError("El correo debe ser texto")
        limpio = valor.strip().lower()
        if len(limpio) > 254 or not cls._PATRON_CORREO.match(limpio):
            raise ClienteInvalidoError(f"Correo inválido '{valor}'")
        return limpio

    @classmethod
    def validar_telefono(cls, valor):
        """(Del Proyecto 2) Revisa el teléfono: 7 a 15 dígitos. Es opcional:
        si viene None, simplemente no se guarda ningún teléfono."""
        if valor is None:
            return None
        if isinstance(valor, bool) or not isinstance(valor, (str, int)):
            raise ClienteInvalidoError("El teléfono debe ser texto o número")
        limpio = str(valor).strip()
        if not cls._PATRON_TELEFONO.match(limpio):
            raise ClienteInvalidoError(
                f"Teléfono inválido '{valor}': debe tener entre 7 y 15 dígitos")
        return limpio

    # ---------------- Getters y setters ----------------
    # "get_" lee el dato privado; "set_" lo cambia, pero SIEMPRE pasando por la validación.
    def get_nombre(self):
        return self.__nombre

    def set_nombre(self, valor):
        self.__nombre = self.validar_nombre(valor)

    def get_documento(self):
        return self.__documento

    def set_documento(self, valor):
        self.__documento = self.validar_documento(valor)

    def get_correo(self):
        return self.__correo

    def set_correo(self, valor):
        self.__correo = self.validar_correo(valor)

    def get_telefono(self):
        return self.__telefono

    def set_telefono(self, valor):
        self.__telefono = self.validar_telefono(valor)

    # Propiedades equivalentes (forma "pythónica"): se usa cliente.nombre en vez de
    # cliente.get_nombre(), y cliente.nombre = "..." en vez de cliente.set_nombre("...").
    nombre = property(get_nombre, set_nombre)
    documento = property(get_documento, set_documento)
    correo = property(get_correo, set_correo)
    telefono = property(get_telefono, set_telefono)

    # ---------------- Métodos de EntidadBase ----------------
    def validar_integridad(self):
        """(Del Proyecto 2) Revisa que los datos guardados sigan siendo válidos."""
        try:
            self.validar_nombre(self.__nombre)
            self.validar_documento(self.__documento)
            self.validar_correo(self.__correo)
            self.validar_telefono(self.__telefono)
            return True
        except ClienteInvalidoError:
            return False

    def mostrar_datos(self):
        """Texto con los datos del cliente."""
        telefono = self.__telefono if self.__telefono else "No registrado"
        return (f"Cliente {self.id} | Nombre: {self.__nombre} | "
                f"Documento: {self.__documento} | Correo: {self.__correo} | "
                f"Teléfono: {telefono}")

## Módulo `servicio.py`

In [ ]:
"""
Módulo: servicio.py
Contiene la clase abstracta Servicio y los tres servicios especializados.

ORIGEN:
    Proyecto 1 -> estructura de clases, FIRMAS de los constructores (sin código
                  manual: el ID se genera solo), encapsulación privada (__),
                  validar_parametros(), cotizar() con "sobrecarga simulada" y
                  rangos de duración por servicio.
    Proyecto 2 -> REGLAS DE COSTO: IVA 19 % con opción aplicar_iva, VideoBeam
                  ($15.000), garantía del equipo, recargo del asesor senior
                  (x1,25), topes de descuento por servicio (50 % / 40 % / 30 %),
                  lista fija de especialidades, especificaciones del equipo,
                  y el bloqueo de disponibilidad al confirmar una reserva.

Nota sobre la sobrecarga: Python no permite dos métodos con el mismo nombre, así
que se simula: cotizar() decide qué variante usar según CUÁNTOS argumentos recibe:
    cotizar(duracion)                        -> con descuento 0 % y con IVA
    cotizar(duracion, descuento)             -> con ese descuento y con IVA
    cotizar(duracion, descuento, aplicar_iva)-> con descuento y decidiendo el IVA
"""

import math                      # Para detectar números raros (infinito, "no es número")
from abc import ABC, abstractmethod



def _es_entero(valor):
    """True si el valor es un número entero de verdad (descarta True/False)."""
    return isinstance(valor, int) and not isinstance(valor, bool)


def _es_numero(valor):
    """True si el valor es un número normal y finito (no bool, no infinito, no NaN)."""
    return (isinstance(valor, (int, float)) and not isinstance(valor, bool)
            and math.isfinite(valor))


# =====================================================================
# Clase abstracta Servicio
# =====================================================================
class Servicio(EntidadBase, ABC):
    """Molde de TODO servicio. Define lo que cada servicio debe saber hacer
    (calcular su costo, describirse y validar sus parámetros). Sus datos son
    privados (__nombre, __tarifa_base, __disponible); las clases hijas los
    consultan solo mediante propiedades."""

    _PREFIJO = "SRV"              # Los IDs serán SRV-001, SRV-002...
    TASA_IVA = 0.19               # (P2) IVA del 19 %
    RANGO_DURACION = (1, 24)      # (P1) Cada hija define su propio rango (mínimo, máximo)
    DESCUENTO_MAXIMO = 0.30       # (P2) Tope de descuento; cada hija define el suyo
    NOMBRE_TIPO = "servicios"     # Solo se usa para escribir mensajes de error claros

    def __init__(self, nombre, tarifa_base, disponible=True):
        # Validaciones de creación: si algo está mal, el servicio NO se crea
        if not isinstance(nombre, str) or not nombre.strip():
            raise ServicioInvalidoError("El nombre del servicio es obligatorio")
        if not _es_numero(tarifa_base) or tarifa_base <= 0:
            raise ServicioInvalidoError(
                f"Tarifa inválida '{tarifa_base}': debe ser un número mayor que 0")
        super().__init__()                        # Asigna el ID (SRV-001...)
        self.__nombre = nombre.strip()            # Quita espacios sobrantes
        self.__tarifa_base = float(tarifa_base)
        self.__disponible = bool(disponible)

    # ---------- Propiedades (encapsulación) ----------
    @property
    def nombre(self):
        return self.__nombre

    @property
    def tarifa_base(self):
        return self.__tarifa_base

    @property
    def disponible(self):
        return self.__disponible

    @disponible.setter
    def disponible(self, valor):
        self.__disponible = bool(valor)

    # ---------- Métodos abstractos: cada servicio los hace a su manera (POLIMORFISMO) ----------
    @abstractmethod
    def calcular_costo(self, duracion, descuento=0.0, aplicar_iva=True, **opciones):
        """Calcula el costo total del servicio."""

    @abstractmethod
    def describir(self):
        """Devuelve un texto que describe el servicio."""

    @abstractmethod
    def validar_parametros(self, duracion, **opciones):
        """Revisa que lleguen los datos necesarios para reservar el servicio."""

    def mostrar_datos(self):
        # Mostrar los datos de un servicio es lo mismo que describirlo
        return self.describir()

    # ---------- Bloqueo de disponibilidad (idea del Proyecto 2) ----------
    # Cuando una reserva se CONFIRMA, el servicio se "bloquea" para que nadie más
    # lo reserve; cuando la reserva se procesa o se cancela, se "libera".
    # AlquilerEquipo los reescribe para trabajar con unidades de stock.
    def bloquear_recurso(self, **opciones):
        """Marca el servicio como ocupado."""
        self.__disponible = False

    def liberar_recurso(self, **opciones):
        """Marca el servicio como libre otra vez."""
        self.__disponible = True

    # ---------- Sobrecarga simulada por cantidad de argumentos (P1, con parámetros del P2) ----------
    @registrar_errores
    def cotizar(self, duracion, *ajustes, **opciones):
        """Calcula un precio de prueba. Los argumentos extra se pasan en orden:
        primero descuento y luego aplicar_iva. Las opciones propias de cada
        servicio (unidades, incluir_garantia...) se envían por nombre."""
        if len(ajustes) > 2:
            raise ParametroInvalidoError(
                "cotizar() admite como máximo descuento y aplicar_iva")
        return self.calcular_costo(duracion, *ajustes, **opciones)

    # ---------- Utilidades compartidas por las clases hijas ----------
    def _validar_duracion(self, duracion):
        """Revisa que la duración sea un número dentro del rango del servicio."""
        minimo, maximo = self.RANGO_DURACION
        if not _es_numero(duracion):
            raise ParametroInvalidoError("La duración debe ser un número")
        if not minimo <= duracion <= maximo:
            raise ParametroInvalidoError(
                f"Duración {duracion} fuera de rango ({minimo} a {maximo})")

    def _validar_ajustes(self, descuento, aplicar_iva):
        """(Regla del P2) Revisa el descuento (entre 0 y el tope del servicio)
        y que aplicar_iva sea True o False."""
        if not _es_numero(descuento):
            raise CalculoInconsistenteError("El descuento debe ser un número")
        if not 0 <= descuento <= self.DESCUENTO_MAXIMO:
            raise CalculoInconsistenteError(
                f"El descuento para {self.NOMBRE_TIPO} debe estar entre 0 % y "
                f"{self.DESCUENTO_MAXIMO:.0%} (se pidió {descuento:.0%})")
        if not isinstance(aplicar_iva, bool):
            raise ParametroInvalidoError("'aplicar_iva' debe ser True o False")

    @staticmethod
    def _verificar_total(total):
        """Última revisión del costo: un total <= 0 o raro significa un cálculo
        inconsistente. Si todo está bien, lo devuelve redondeado a 2 decimales."""
        if not math.isfinite(total) or total <= 0:
            raise CalculoInconsistenteError(f"Resultado inconsistente: {total}")
        return round(total, 2)


# =====================================================================
# ReservaSala
# =====================================================================
class ReservaSala(Servicio):
    """Servicio de reserva de salas. La duración se mide en HORAS."""

    RANGO_DURACION = (1, 12)      # (P1) De 1 a 12 horas
    DESCUENTO_MAXIMO = 0.50       # (P2) Máximo 50 % de descuento
    NOMBRE_TIPO = "salas"
    COSTO_VIDEOBEAM = 15000.0     # (P2) Cargo fijo si la sala incluye VideoBeam
    CAPACIDAD_MAXIMA = 100        # (P2) Una sala admite como máximo 100 personas

    @registrar_errores
    def __init__(self, nombre, tarifa_hora, capacidad, disponible=True, tiene_videobeam=True):
        # Firma del P1: (nombre, tarifa_hora, capacidad, disponible). Se agregó al final
        # "tiene_videobeam" (idea del P2) con valor por defecto, para no romper la firma.
        if not _es_entero(capacidad) or not 1 <= capacidad <= self.CAPACIDAD_MAXIMA:
            raise ServicioInvalidoError(
                f"La capacidad debe ser un entero entre 1 y {self.CAPACIDAD_MAXIMA}")
        if not isinstance(tiene_videobeam, bool):
            raise ServicioInvalidoError("'tiene_videobeam' debe ser True o False")
        super().__init__(nombre, tarifa_hora, disponible)
        self.__capacidad = capacidad
        self.__tiene_videobeam = tiene_videobeam

    @property
    def capacidad(self):
        return self.__capacidad

    @property
    def tiene_videobeam(self):
        return self.__tiene_videobeam

    def calcular_costo(self, duracion, descuento=0.0, aplicar_iva=True, **opciones):
        """REGLA DEL P2: (tarifa x horas + VideoBeam) -> se resta el descuento -> se suma el IVA."""
        self._validar_duracion(duracion)
        self._validar_ajustes(descuento, aplicar_iva)
        subtotal = self.tarifa_base * duracion          # Valor de las horas
        if self.__tiene_videobeam:
            subtotal += self.COSTO_VIDEOBEAM            # Suma el VideoBeam
        con_descuento = subtotal * (1 - descuento)      # Resta el descuento
        # Suma el IVA solo si se pidió
        total = con_descuento * (1 + self.TASA_IVA) if aplicar_iva else con_descuento
        return self._verificar_total(total)

    def describir(self):
        video = "con VideoBeam" if self.__tiene_videobeam else "sin VideoBeam"
        return (f"Sala '{self.nombre}' [{self.id}] | Capacidad: {self.__capacidad} personas | "
                f"{video} | Tarifa: ${self.tarifa_base:,.0f}/hora")

    def validar_parametros(self, duracion, asistentes=None, **opciones):
        """Para reservar una sala hay que indicar cuántos asistentes irán."""
        if asistentes is None:
            raise ParametroFaltanteError("Falta el parámetro obligatorio 'asistentes'")
        self._validar_duracion(duracion)
        if not _es_entero(asistentes) or asistentes <= 0:
            raise ParametroInvalidoError("'asistentes' debe ser un entero > 0")
        if asistentes > self.__capacidad:
            raise ParametroInvalidoError(
                f"{asistentes} asistentes superan la capacidad ({self.__capacidad})")
        return True

    def validar_integridad(self):
        """(P2) Revisa que los datos de la sala sigan siendo coherentes."""
        return self.tarifa_base > 0 and 1 <= self.__capacidad <= self.CAPACIDAD_MAXIMA


# =====================================================================
# AlquilerEquipo
# =====================================================================
class AlquilerEquipo(Servicio):
    """Alquiler de equipos. La duración se mide en DÍAS. Cada equipo tiene un
    stock (cuántas unidades hay) que baja al confirmar y sube al devolver."""

    RANGO_DURACION = (1, 30)      # (P1) De 1 a 30 días
    DESCUENTO_MAXIMO = 0.40       # (P2) Máximo 40 % de descuento
    NOMBRE_TIPO = "equipos"
    MIN_ESPECIFICACIONES = 5      # (P2) Las especificaciones deben tener 5+ caracteres

    @registrar_errores
    def __init__(self, nombre, tarifa_dia, tipo_equipo, stock, disponible=True,
                 especificaciones=None, deposito_garantia=0.0):
        # Firma del P1: (nombre, tarifa_dia, tipo_equipo, stock, disponible).
        # Se agregaron al final, opcionales, "especificaciones" y "deposito_garantia" (del P2).
        if not isinstance(tipo_equipo, str) or not tipo_equipo.strip():
            raise ServicioInvalidoError("El tipo de equipo es obligatorio")
        if not _es_entero(stock) or stock <= 0:
            raise ServicioInvalidoError("El stock debe ser un entero > 0")
        if especificaciones is None:
            especificaciones = tipo_equipo           # Si no las dan, usa el tipo de equipo
        if (not isinstance(especificaciones, str)
                or len(especificaciones.strip()) < self.MIN_ESPECIFICACIONES):
            raise ServicioInvalidoError(
                f"Las especificaciones deben tener al menos {self.MIN_ESPECIFICACIONES} caracteres")
        if not _es_numero(deposito_garantia) or deposito_garantia < 0:
            raise ServicioInvalidoError("El depósito de garantía debe ser un número >= 0")
        super().__init__(nombre, tarifa_dia, disponible)
        self.__tipo_equipo = tipo_equipo.strip()
        self.__stock = stock
        self.__especificaciones = especificaciones.strip()
        self.__deposito_garantia = float(deposito_garantia)

    @property
    def tipo_equipo(self):
        return self.__tipo_equipo

    @property
    def stock(self):
        return self.__stock

    @property
    def especificaciones(self):
        return self.__especificaciones

    @property
    def deposito_garantia(self):
        return self.__deposito_garantia

    def calcular_costo(self, duracion, descuento=0.0, aplicar_iva=True,
                       unidades=1, incluir_garantia=True, **opciones):
        """REGLA DEL P2: alquiler = tarifa x días (x unidades) -> se resta el descuento ->
        el IVA se calcula sobre el alquiler -> al final se suma la garantía (que NO paga IVA).
        Con unidades=1 el resultado es idéntico al del Proyecto 2."""
        self._validar_duracion(duracion)
        self._validar_ajustes(descuento, aplicar_iva)
        if not _es_entero(unidades) or unidades < 1:
            raise ParametroInvalidoError("'unidades' debe ser un entero >= 1")
        if not isinstance(incluir_garantia, bool):
            raise ParametroInvalidoError("'incluir_garantia' debe ser True o False")
        alquiler = self.tarifa_base * duracion * unidades    # Valor del alquiler
        con_descuento = alquiler * (1 - descuento)           # Resta el descuento
        iva = con_descuento * self.TASA_IVA if aplicar_iva else 0.0   # IVA solo del alquiler
        # La garantía es un depósito por cada unidad alquilada
        garantia = self.__deposito_garantia * unidades if incluir_garantia else 0.0
        return self._verificar_total(con_descuento + iva + garantia)

    def describir(self):
        return (f"Equipo '{self.nombre}' [{self.id}] | Tipo: {self.__tipo_equipo} | "
                f"Specs: {self.__especificaciones} | Stock: {self.__stock} | "
                f"Tarifa: ${self.tarifa_base:,.0f}/día | "
                f"Garantía: ${self.__deposito_garantia:,.0f}")

    def validar_parametros(self, duracion, unidades=None, **opciones):
        """Para alquilar hay que indicar cuántas unidades, y deben alcanzar en stock."""
        if unidades is None:
            raise ParametroFaltanteError("Falta el parámetro obligatorio 'unidades'")
        self._validar_duracion(duracion)
        if not _es_entero(unidades) or unidades <= 0:
            raise ParametroInvalidoError("'unidades' debe ser un entero > 0")
        if unidades > self.__stock:
            raise ParametroInvalidoError(
                f"Se piden {unidades} unidades y solo hay {self.__stock} en stock")
        return True

    # Un equipo no se bloquea entero: solo se descuentan las unidades alquiladas.
    def bloquear_recurso(self, unidades=1, **opciones):
        """Al confirmar: resta las unidades del stock. Si no queda nada, se apaga."""
        self.__stock -= unidades
        if self.__stock <= 0:
            self.disponible = False

    def liberar_recurso(self, unidades=1, **opciones):
        """Al devolver el equipo: suma las unidades al stock y vuelve a estar disponible."""
        self.__stock += unidades
        self.disponible = True

    def validar_integridad(self):
        """(P2) Revisa que los datos del equipo sigan siendo coherentes."""
        return self.tarifa_base > 0 and self.__stock >= 0 and self.__deposito_garantia >= 0


# =====================================================================
# AsesoriaEspecializada
# =====================================================================
class AsesoriaEspecializada(Servicio):
    """Asesoría por horas. El nivel del asesor (junior/senior) cambia el costo."""

    RANGO_DURACION = (1, 8)       # (P1) De 1 a 8 horas
    DESCUENTO_MAXIMO = 0.30       # (P2) Máximo 30 % de descuento
    NOMBRE_TIPO = "asesorías"
    NIVELES = {"junior": 1.0, "senior": 1.25}    # (P2) El senior cuesta 25 % más
    # (P2) Solo se aceptan estas especialidades
    ESPECIALIDADES_VALIDAS = ("ARQUITECTURA", "CIBERSEGURIDAD", "CLOUD", "DATOS", "DEVOPS")

    @registrar_errores
    def __init__(self, nombre, tarifa_hora, especialidad, nivel="junior", disponible=True):
        # Firma idéntica a la del Proyecto 1.
        if not isinstance(especialidad, str) or not especialidad.strip():
            raise ServicioInvalidoError("La especialidad es obligatoria")
        especialidad = especialidad.strip().upper()   # "cloud" y "Cloud" valen igual
        if especialidad not in self.ESPECIALIDADES_VALIDAS:
            raise ServicioInvalidoError(
                f"Especialidad '{especialidad}' inválida. "
                f"Opciones: {', '.join(self.ESPECIALIDADES_VALIDAS)}")
        if nivel not in self.NIVELES:
            raise ServicioInvalidoError(
                f"Nivel '{nivel}' inválido. Opciones: {', '.join(self.NIVELES)}")
        super().__init__(nombre, tarifa_hora, disponible)
        self.__especialidad = especialidad
        self.__nivel = nivel

    @property
    def especialidad(self):
        return self.__especialidad

    @property
    def nivel(self):
        return self.__nivel

    def calcular_costo(self, duracion, descuento=0.0, aplicar_iva=True, **opciones):
        """REGLA DEL P2: tarifa x horas x factor del nivel (senior = 1,25) ->
        se resta el descuento -> se suma el IVA."""
        self._validar_duracion(duracion)
        self._validar_ajustes(descuento, aplicar_iva)
        subtotal = self.tarifa_base * duracion * self.NIVELES[self.__nivel]
        con_descuento = subtotal * (1 - descuento)
        total = con_descuento * (1 + self.TASA_IVA) if aplicar_iva else con_descuento
        return self._verificar_total(total)

    def describir(self):
        return (f"Asesoría '{self.nombre}' [{self.id}] | Especialidad: {self.__especialidad} | "
                f"Nivel: {self.__nivel} | Tarifa: ${self.tarifa_base:,.0f}/hora")

    def validar_parametros(self, duracion, **opciones):
        """Una asesoría solo necesita una duración válida."""
        self._validar_duracion(duracion)
        return True

    def validar_integridad(self):
        """(P2) Revisa que los datos de la asesoría sigan siendo coherentes."""
        return (self.tarifa_base > 0 and self.__nivel in self.NIVELES
                and self.__especialidad in self.ESPECIALIDADES_VALIDAS)

## Módulo `reserva.py`

In [ ]:
"""
Módulo: reserva.py
Clase Reserva: une un cliente, un servicio, una duración y un estado.

ORIGEN:
    Proyecto 1 -> firma Reserva(cliente, servicio, duracion, **parametros), estados,
                  try/except/else/finally en confirmar/cancelar/procesar,
                  encadenamiento "raise ... from", decorador de logs.
    Proyecto 2 -> el COSTO se calcula al CREAR la reserva (así un descuento ilegal se
                  detecta de inmediato), el servicio se BLOQUEA al confirmar y se
                  LIBERA al procesar/cancelar, y no se puede reservar un servicio
                  que ya está ocupado.

Ciclo de vida:
    PENDIENTE --confirmar()--> CONFIRMADA --procesar()--> PROCESADA
        |                          |
        +-------cancelar()---------+--> CANCELADA
"""



class Reserva(EntidadBase):
    """Reserva de un servicio por parte de un cliente. Sus datos son privados:
    el estado solo cambia con confirmar(), cancelar() y procesar(); nunca se
    puede asignar directamente desde afuera."""

    _PREFIJO = "RES"   # Los IDs serán RES-001, RES-002...

    # Los cuatro estados posibles
    PENDIENTE = "PENDIENTE"
    CONFIRMADA = "CONFIRMADA"
    CANCELADA = "CANCELADA"
    PROCESADA = "PROCESADA"

    @registrar_errores
    def __init__(self, cliente, servicio, duracion, descuento=0.0, aplicar_iva=True,
                 **parametros):
        """parametros = datos propios del servicio (asistentes, unidades,
        incluir_garantia...). Se pasan por nombre: Reserva(c, s, 3, asistentes=8)."""

        # 1) Verifica que los objetos sean del tipo correcto
        if not isinstance(cliente, Cliente):
            raise ReservaError("La reserva requiere un Cliente válido")
        if not isinstance(servicio, Servicio):
            raise ReservaError("La reserva requiere un Servicio válido")

        # 2) (P2) Si el servicio está ocupado o apagado, ni siquiera se crea la reserva
        if not servicio.disponible:
            raise ServicioNoDisponibleError(
                f"El servicio '{servicio.nombre}' no está disponible")

        # 3) El servicio revisa sus parámetros (P1) y calcula el costo (P2)
        try:
            servicio.validar_parametros(duracion, **parametros)
            costo = servicio.calcular_costo(duracion, descuento, aplicar_iva, **parametros)
        except SoftwareFJError as error:
            # Encadenamiento: se guarda el error original como "causa" del nuevo
            raise ReservaError(
                f"No se pudo crear la reserva para {servicio.nombre}: {error}") from error

        # 4) Todo está bien: ahora sí se asigna el ID y se guardan los datos privados
        super().__init__()
        self.__cliente = cliente
        self.__servicio = servicio
        self.__duracion = duracion
        self.__descuento = descuento
        self.__aplicar_iva = aplicar_iva
        self.__parametros = dict(parametros)   # Se guarda una COPIA, por seguridad
        self.__estado = self.PENDIENTE         # Toda reserva nace pendiente
        self.__costo_total = costo
        Logger.registrar_evento(
            f"Reserva {self.id} creada: cliente {cliente.id}, servicio {servicio.id}, "
            f"total ${costo:,.2f}")

    # ---------- Propiedades de solo lectura ----------
    @property
    def cliente(self):
        return self.__cliente

    @property
    def servicio(self):
        return self.__servicio

    @property
    def duracion(self):
        return self.__duracion

    @property
    def descuento(self):
        return self.__descuento

    @property
    def aplicar_iva(self):
        return self.__aplicar_iva

    @property
    def parametros(self):
        return dict(self.__parametros)    # Entrega una copia, no el original

    @property
    def estado(self):
        return self.__estado

    @property
    def costo_total(self):
        return self.__costo_total

    # ---------- Operaciones ----------
    @registrar_errores
    def confirmar(self):
        """PENDIENTE -> CONFIRMADA. Bloquea el servicio. Usa try/except/else/finally."""
        try:
            # Solo se confirma lo que está pendiente
            if self.__estado != self.PENDIENTE:
                raise OperacionNoPermitidaError(
                    f"Solo se confirma una reserva PENDIENTE (estado actual: {self.__estado})")
            # El servicio pudo haber sido tomado por otra reserva mientras tanto
            if not self.__servicio.disponible:
                raise ServicioNoDisponibleError(
                    f"El servicio '{self.__servicio.nombre}' no está disponible")
            # Vuelve a validar (por ejemplo, que todavía alcance el stock del equipo)
            self.__servicio.validar_parametros(self.__duracion, **self.__parametros)
        except SoftwareFJError as error:
            raise ReservaError(f"No se pudo confirmar la reserva {self.id}") from error
        else:
            # "else" se ejecuta SOLO si arriba no hubo ningún error
            self.__servicio.bloquear_recurso(**self.__parametros)   # (P2) Bloquea el servicio
            self.__estado = self.CONFIRMADA
            Logger.registrar_evento(f"Reserva {self.id} confirmada")
            return True
        finally:
            # "finally" se ejecuta SIEMPRE, haya habido error o no
            Logger.registrar_evento(f"Fin del intento de confirmar la reserva {self.id}")

    @registrar_errores
    def cancelar(self):
        """PENDIENTE o CONFIRMADA -> CANCELADA. Si estaba confirmada, libera el servicio."""
        try:
            if self.__estado in (self.CANCELADA, self.PROCESADA):
                raise OperacionNoPermitidaError(
                    f"No se puede cancelar una reserva {self.__estado}")
        except OperacionNoPermitidaError as error:
            raise ReservaError(f"No se pudo cancelar la reserva {self.id}") from error
        else:
            # Solo una reserva CONFIRMADA tiene el servicio bloqueado. Una pendiente
            # no bloqueó nada, así que no hay nada que liberar (evita liberar de más).
            if self.__estado == self.CONFIRMADA:
                self.__servicio.liberar_recurso(**self.__parametros)
            self.__estado = self.CANCELADA
            Logger.registrar_evento(f"Reserva {self.id} cancelada")
            return True
        finally:
            Logger.registrar_evento(f"Fin del intento de cancelar la reserva {self.id}")

    @registrar_errores
    def procesar(self):
        """CONFIRMADA -> PROCESADA. Cierra la reserva, libera el servicio y devuelve
        el costo total (que ya se calculó al crear la reserva)."""
        try:
            if self.__estado != self.CONFIRMADA:
                raise OperacionNoPermitidaError(
                    f"Solo se procesa una reserva CONFIRMADA (estado actual: {self.__estado})")
        except OperacionNoPermitidaError as error:
            raise ReservaError(f"No se pudo procesar la reserva {self.id}") from error
        else:
            self.__servicio.liberar_recurso(**self.__parametros)    # (P2) Libera el servicio
            self.__estado = self.PROCESADA
            Logger.registrar_evento(
                f"Reserva {self.id} procesada. Total: ${self.__costo_total:,.2f}")
            return self.__costo_total
        finally:
            Logger.registrar_evento(f"Fin del intento de procesar la reserva {self.id}")

    # ---------- Métodos de EntidadBase ----------
    def validar_integridad(self):
        """(P2) Revisa que la reserva sea coherente: estado válido y costo positivo."""
        estados = (self.PENDIENTE, self.CONFIRMADA, self.CANCELADA, self.PROCESADA)
        return self.__estado in estados and self.__costo_total > 0

    def mostrar_reserva(self):
        """Texto con todos los datos de la reserva."""
        iva = "con IVA" if self.__aplicar_iva else "sin IVA"
        return (f"Reserva {self.id} | Cliente: {self.__cliente.nombre} | "
                f"Servicio: {self.__servicio.nombre} | Duración: {self.__duracion} | "
                f"Descuento: {self.__descuento:.0%} | {iva} | "
                f"Estado: {self.__estado} | Total: ${self.__costo_total:,.2f}")

    def mostrar_datos(self):
        return self.mostrar_reserva()

## Módulo `gestor.py`

In [ ]:
"""
Módulo: gestor.py
ORIGEN: Proyecto 2 -> la IDEA de una clase GestorSoftwareFJ que guarda las listas
                      de clientes, servicios y reservas (en el Proyecto 1 eran
                      listas sueltas y globales dentro de simulacion.py).
        Proyecto 1 -> bloqueo de documentos repetidos (ClienteDuplicadoError),
                      listas privadas y decorador de logs.

Aquí NO hay base de datos: todo vive en listas dentro de la memoria del programa.
"""



class GestorSoftwareFJ:
    """Guarda y organiza los clientes, servicios y reservas del sistema."""

    def __init__(self):
        # Listas PRIVADAS: desde afuera no se pueden modificar directamente
        self.__clientes = []
        self.__servicios = []
        self.__reservas = []
        Logger.registrar_evento("Gestor Software FJ iniciado (datos en memoria)")

    # ---------- Consulta (devuelven tuplas: copias que no se pueden alterar) ----------
    @property
    def clientes(self):
        return tuple(self.__clientes)

    @property
    def servicios(self):
        return tuple(self.__servicios)

    @property
    def reservas(self):
        return tuple(self.__reservas)

    # ---------- Registro ----------
    @registrar_errores
    def registrar_cliente(self, cliente):
        """Agrega un cliente a la lista. No permite documentos repetidos."""
        if not isinstance(cliente, Cliente):
            raise ClienteInvalidoError("Solo se pueden registrar objetos de tipo Cliente")
        for existente in self.__clientes:
            if existente.documento == cliente.documento:     # ¿Ya existe ese documento?
                raise ClienteDuplicadoError(
                    f"Ya existe un cliente con el documento {cliente.documento} "
                    f"({existente.id})")
        self.__clientes.append(cliente)
        Logger.registrar_evento(f"Cliente {cliente.id} registrado")
        return cliente

    @registrar_errores
    def registrar_servicio(self, servicio):
        """Agrega un servicio a la lista (cuida que no se registre dos veces)."""
        if not isinstance(servicio, Servicio):
            raise ServicioInvalidoError("Solo se pueden registrar objetos de tipo Servicio")
        if any(existente.id == servicio.id for existente in self.__servicios):
            raise ServicioInvalidoError(f"El servicio {servicio.id} ya está registrado")
        self.__servicios.append(servicio)
        Logger.registrar_evento(f"Servicio {servicio.id} registrado")
        return servicio

    @registrar_errores
    def registrar_reserva(self, reserva):
        """Agrega una reserva a la lista (cuida que no se registre dos veces)."""
        if not isinstance(reserva, Reserva):
            raise ReservaError("Solo se pueden registrar objetos de tipo Reserva")
        if any(existente.id == reserva.id for existente in self.__reservas):
            raise ReservaError(f"La reserva {reserva.id} ya está registrada")
        self.__reservas.append(reserva)
        return reserva

    # ---------- Búsqueda ----------
    def buscar_cliente_por_documento(self, documento):
        """Devuelve el cliente con ese documento, o None si no existe."""
        documento = str(documento).strip()
        for cliente in self.__clientes:
            if cliente.documento == documento:
                return cliente
        return None

    def resumen(self):
        """Cantidades registradas, en un diccionario."""
        return {"clientes": len(self.__clientes),
                "servicios": len(self.__servicios),
                "reservas": len(self.__reservas)}

## Módulo `simulacion.py`

In [ ]:
"""
Módulo: simulacion.py
Ejecuta 15 operaciones (6 válidas y 9 no válidas) usando el GestorSoftwareFJ.

ORIGEN:
    Proyecto 1 -> el mecanismo _ejecutar() (cada operación es independiente: si una
                  falla se anota en el log y el programa SIGUE con la siguiente) y
                  la mayoría de operaciones.
    Proyecto 2 -> los valores y casos de demostración (sala con VideoBeam y descuento,
                  equipo con garantía, asesoría senior sin IVA, servicio ocupado,
                  descuento ilegal, cancelar una reserva ya procesada) y el uso del
                  gestor en lugar de listas globales.

Cada operación se crea sus propios datos si los necesita (funciones _cliente y
_servicio). Así, si una operación falla, las demás no se ven afectadas.
"""


# Datos de los clientes de demostración: (nombre, documento, correo, teléfono)
DATOS_CLIENTES = (
    ("Ana María Pérez", "1012345678", "ana.perez@correo.com", "3104567890"),
    ("Carlos Gómez", 80123456, "carlos.gomez@correo.com", "3159876543"),
)

# "Fábricas" de servicios: cada una es una mini-función que crea el servicio cuando se necesita.
# Los valores salen del Proyecto 2; las firmas (sin código manual) son las del Proyecto 1.
_FABRICAS_SERVICIOS = {
    "sala": lambda: ReservaSala("Sala Ejecutiva Santander", 80000, 20, tiene_videobeam=True),
    "sala_aux": lambda: ReservaSala("Sala Auditorio", 120000, 50, tiene_videobeam=False),
    "equipo": lambda: AlquilerEquipo(
        "Workstation Dell Precision", 65000, "Workstation", 5,
        especificaciones="Intel Core i9, 64GB RAM, RTX 4080", deposito_garantia=200000),
    "asesoria": lambda: AsesoriaEspecializada(
        "Auditoría Arquitectura Cloud", 150000, "Cloud", "senior"),
    # Este proyector se crea YA apagado (disponible=False) para probar el error
    "proyector": lambda: AlquilerEquipo("Proyector 4K", 30000, "Proyector", 2,
                                        disponible=False),
}
_servicios_demo = {}   # Aquí se guardan los servicios ya creados, para reutilizarlos


# Funciones auxiliares
def _cliente(gestor, indice):
    """Devuelve el cliente de demostración (0 o 1). Si no existe todavía, lo crea y lo registra."""
    nombre, documento, correo, telefono = DATOS_CLIENTES[indice]
    existente = gestor.buscar_cliente_por_documento(documento)
    if existente is not None:
        return existente
    return gestor.registrar_cliente(Cliente(nombre, documento, correo, telefono))


def _servicio(gestor, clave):
    """Devuelve el servicio de demostración indicado. Si no existe todavía, lo crea y lo registra."""
    if clave not in _servicios_demo:
        _servicios_demo[clave] = gestor.registrar_servicio(_FABRICAS_SERVICIOS[clave]())
    return _servicios_demo[clave]


def _nueva_reserva(gestor, cliente, servicio, duracion, **opciones):
    """Crea una reserva y la guarda en el gestor.
    opciones puede traer descuento, aplicar_iva, asistentes, unidades..."""
    return gestor.registrar_reserva(Reserva(cliente, servicio, duracion, **opciones))


# Ejecución controlada de cada operación
def _ejecutar(numero, tipo, descripcion, accion):
    """Ejecuta UNA operación y controla cualquier error para que el programa no se caiga.
    Devuelve True si terminó bien y False si falló."""
    print(f"\n[{numero:02d}] ({tipo}) {descripcion}")
    try:
        resultado = accion()                       # Corre la operación
    except SoftwareFJError as error:
        # Error propio del sistema: se muestra y el programa continúa.
        # Si la clase ya lo anotó en el log, aquí solo se deja una advertencia.
        if error.registrada:
            Logger.registrar_advertencia(
                f"Operación {numero} no completada - {descripcion}")
        else:
            Logger.registrar_error(f"Operación {numero} fallida - {descripcion}", error)
        print(f"     X Error controlado: {error}")
        if error.__cause__ is not None:            # Si el error venía de otro, lo muestra
            print(f"       Causa original: {error.__cause__}")
        return False
    except Exception as error:                     # Cualquier error imprevisto también se contiene
        Logger.registrar_error(f"Operación {numero} - error inesperado", error)
        print(f"     X Error inesperado controlado: {error}")
        return False
    else:
        # Solo si no hubo errores
        Logger.registrar_evento(f"Operación {numero} exitosa - {descripcion}")
        print(f"     OK {resultado}")
        return True
    finally:
        # Siempre, con o sin error
        Logger.registrar_evento(f"Operación {numero} finalizada")


def ejecutar_simulaciones(gestor):
    """Corre las 15 operaciones sobre el gestor recibido y devuelve un resumen."""
    _servicios_demo.clear()    # Permite ejecutar la simulación varias veces sin mezclar datos

    Logger.registrar_evento("===== INICIO DE LA SIMULACIÓN =====")

    # ------------------------- OPERACIONES VÁLIDAS -------------------------
    def v1():  # Registrar clientes válidos (ahora con teléfono)
        for datos in DATOS_CLIENTES:
            gestor.registrar_cliente(Cliente(*datos))   # El * reparte la tupla en argumentos
        return " / ".join(c.mostrar_datos() for c in gestor.clientes)

    def v2():  # Sala con VideoBeam, 4 horas, 10 % de descuento -> esperado $358.785,00
        reserva = _nueva_reserva(gestor, _cliente(gestor, 0), _servicio(gestor, "sala"), 4,
                                 descuento=0.10, asistentes=8)
        reserva.confirmar()
        reserva.procesar()
        return reserva.mostrar_reserva()

    def v3():  # Equipo 5 días, 5 % de descuento, con garantía -> esperado $567.412,50
        equipo = _servicio(gestor, "equipo")
        reserva = _nueva_reserva(gestor, _cliente(gestor, 0), equipo, 5,
                                 descuento=0.05, unidades=1)
        reserva.confirmar()
        stock_alquilado = equipo.stock       # Al confirmar, el stock baja de 5 a 4
        reserva.procesar()                   # Al procesar, la unidad regresa: stock vuelve a 5
        return (f"{reserva.mostrar_reserva()} | Stock durante el alquiler: "
                f"{stock_alquilado} | Stock al devolver: {equipo.stock}")

    def v4():  # Asesoría senior, 6 horas, SIN IVA -> esperado $1.125.000,00
        reserva = _nueva_reserva(gestor, _cliente(gestor, 1), _servicio(gestor, "asesoria"), 6,
                                 aplicar_iva=False)
        reserva.confirmar()
        reserva.procesar()
        return reserva.mostrar_reserva()

    def v5():  # Reservar, confirmar y cancelar: el servicio se bloquea y luego se libera
        asesoria = _servicio(gestor, "asesoria")
        reserva = _nueva_reserva(gestor, _cliente(gestor, 1), asesoria, 2)
        reserva.confirmar()
        disponible_confirmada = asesoria.disponible   # Debe ser False (está bloqueada)
        reserva.cancelar()
        return (f"{reserva.mostrar_reserva()} | Disponible al confirmar: "
                f"{disponible_confirmada} | Disponible al cancelar: {asesoria.disponible}")

    def v6():  # Sobrecarga simulada: tres formas de cotizar la misma sala
        sala = _servicio(gestor, "sala")
        base = sala.cotizar(2)                  # Solo duración: sin descuento y con IVA
        con_descuento = sala.cotizar(2, 0.10)   # Duración + 10 % de descuento
        completo = sala.cotizar(2, 0.10, False) # Duración + 10 % de descuento + SIN IVA
        return (f"Sala 2 h -> básico: ${base:,.2f} | 10 % desc.: ${con_descuento:,.2f} | "
                f"10 % desc. sin IVA: ${completo:,.2f}")

    # ----------------------- OPERACIONES NO VÁLIDAS ------------------------
    def i1():  # Cliente con nombre, documento, correo y teléfono inválidos
        return gestor.registrar_cliente(Cliente("L1", "12AB", "luis@@correo", "123"))

    def i2():  # Servicio mal creado: tarifa negativa
        return ReservaSala("Sala VIP", -50000, 10)

    def i3():  # Falta el parámetro obligatorio 'asistentes'
        return _nueva_reserva(gestor, _cliente(gestor, 0), _servicio(gestor, "sala"), 2)

    def i4():  # Servicio apagado: el proyector está no disponible
        return _nueva_reserva(gestor, _cliente(gestor, 0), _servicio(gestor, "proyector"), 1,
                              unidades=1)

    def i5():  # Servicio ocupado: la primera reserva lo bloquea, la segunda no puede
        auditorio = _servicio(gestor, "sala_aux")
        primera = _nueva_reserva(gestor, _cliente(gestor, 0), auditorio, 2, asistentes=10)
        primera.confirmar()     # El auditorio queda ocupado
        return _nueva_reserva(gestor, _cliente(gestor, 1), auditorio, 1, asistentes=5)

    def i6():  # Operación no permitida: procesar una reserva cancelada
        reserva = _nueva_reserva(gestor, _cliente(gestor, 1), _servicio(gestor, "sala"), 1,
                                 asistentes=3)
        reserva.cancelar()
        return reserva.procesar()

    def i7():  # Descuento ilegal: 85 % en una asesoría (el tope es 30 %)
        return _nueva_reserva(gestor, _cliente(gestor, 0), _servicio(gestor, "asesoria"), 3,
                              descuento=0.85)

    def i8():  # Operación no permitida: cancelar una reserva que ya fue procesada
        reserva = _nueva_reserva(gestor, _cliente(gestor, 1), _servicio(gestor, "asesoria"), 1)
        reserva.confirmar()
        reserva.procesar()
        return reserva.cancelar()

    def i9():  # Cliente duplicado: mismo documento que Ana María
        return gestor.registrar_cliente(
            Cliente("Ana María Pérez", "1012345678", "otro.correo@correo.com", "3001112233"))

    # Lista con todas las operaciones: (tipo, descripción, función)
    operaciones = [
        ("VÁLIDA", "Registrar clientes válidos (con teléfono)", v1),
        ("VÁLIDA", "Sala con VideoBeam, 4 h y 10 % de descuento", v2),
        ("VÁLIDA", "Alquilar equipo con garantía (el stock baja y sube)", v3),
        ("VÁLIDA", "Asesoría senior de 6 h sin IVA", v4),
        ("VÁLIDA", "Crear, confirmar y cancelar (el servicio se libera)", v5),
        ("VÁLIDA", "Cotizar una sala de tres formas (sobrecarga)", v6),
        ("NO VÁLIDA", "Registrar cliente con datos inválidos", i1),
        ("NO VÁLIDA", "Crear sala con tarifa negativa", i2),
        ("NO VÁLIDA", "Reservar sala sin indicar asistentes", i3),
        ("NO VÁLIDA", "Reservar un servicio no disponible", i4),
        ("NO VÁLIDA", "Reservar una sala que ya está ocupada", i5),
        ("NO VÁLIDA", "Procesar una reserva cancelada", i6),
        ("NO VÁLIDA", "Asesoría con descuento del 85 % (cálculo inconsistente)", i7),
        ("NO VÁLIDA", "Cancelar una reserva ya procesada", i8),
        ("NO VÁLIDA", "Registrar un cliente con documento repetido", i9),
    ]

    resumen = {"exitosas": 0, "fallidas": 0}
    # enumerate(..., start=1) numera las operaciones desde 1
    for numero, (tipo, descripcion, accion) in enumerate(operaciones, start=1):
        if _ejecutar(numero, tipo, descripcion, accion):
            resumen["exitosas"] += 1
        else:
            resumen["fallidas"] += 1

    Logger.registrar_evento(
        f"===== FIN DE LA SIMULACIÓN: {resumen['exitosas']} exitosas, "
        f"{resumen['fallidas']} fallidas =====")
    return resumen

## Módulo `main.py`

In [ ]:
"""
Módulo: main.py
Punto de entrada del sistema. Crea el gestor, corre la simulación y muestra
el resumen final.

ORIGEN: Proyecto 1 (estructura de main, red de seguridad para errores y resumen)
        + Proyecto 2 (uso de GestorSoftwareFJ para guardar los datos).

Ejecución:  python main.py
"""



def main():
    """Prepara el logger, corre la simulación y muestra el resumen final."""
    Logger.configurar_logger()   # Deja listo el archivo de logs
    print("=" * 70)
    print(" SOFTWARE FJ - Sistema Integral de Gestión de Clientes, Servicios y Reservas")
    print("=" * 70)

    gestor = GestorSoftwareFJ()  # Aquí vivirán clientes, servicios y reservas

    try:
        resumen = ejecutar_simulaciones(gestor)
    except KeyboardInterrupt:
        # El usuario presionó Ctrl+C: se sale con elegancia
        Logger.registrar_evento("Ejecución interrumpida por el usuario")
        print("\nEjecución interrumpida por el usuario.")
        return
    except Exception as error:
        # Última red de seguridad: si algo inesperado pasa, el programa no se cae feo
        Logger.registrar_error("Error crítico en la ejecución principal", error)
        print(f"Error crítico controlado: {error}")
        return
    finally:
        print("\n" + "-" * 70)

    # Resumen final usando los datos del gestor
    totales = gestor.resumen()
    print("RESUMEN FINAL")
    print(f"  Operaciones exitosas : {resumen['exitosas']}")
    print(f"  Operaciones fallidas : {resumen['fallidas']} (controladas, el sistema siguió activo)")
    print(f"  Clientes registrados : {totales['clientes']}")
    print(f"  Servicios creados    : {totales['servicios']}")
    print(f"  Reservas creadas     : {totales['reservas']}")
    for reserva in gestor.reservas:
        print(f"    - {reserva.mostrar_reserva()}")
    print("  Estado final de los servicios:")
    for servicio in gestor.servicios:
        estado = "disponible" if servicio.disponible else "NO disponible"
        print(f"    - {servicio.nombre} [{servicio.id}]: {estado}")
    print(f"\nRegistro completo de eventos y errores en: {Logger.RUTA_LOG}")


if __name__ == "__main__":
    main()